# Train + persist the final delivery-delay GLM model

Single-purpose notebook: trains the validated clean GLM baseline (checkout-time
framing, VIF-cleaned features, QR solver, ridge disabled) and saves it as a
**permanently named** model (`DELIVERY_DELAY_GLM_FINAL`) so it survives past this
notebook session -- OML4Py models built without an explicit `model_name` are
temporary and get dropped when the connection ends.

Full experimentation history (dead ends, ml-engineer reviews, the solver/VIF/
row-alignment investigations) lives in `ml/delivery_delay/delivery_delay_oml4py.ipynb`
-- this notebook is just the clean, final training step, meant to be run once.

In [ ]:
import oml
import pandas as pd
from sklearn.preprocessing import PowerTransformer
from sklearn.metrics import mean_squared_error, r2_score

# Fresh pull + time-based split
delay_features = oml.sync(table='DELIVERY_DELAY_FEATURES', schema='OML_USER')
df = delay_features.pull()
df = df.sort_values('ORDER_PURCHASE_TIMESTAMP').reset_index(drop=True)

split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()
print(f'Train: {len(train_df)}, Test: {len(test_df)}')

In [ ]:
# Yeo-Johnson on target, fit on train only
pt = PowerTransformer(method='yeo-johnson')
train_df['DELAY_TRANSFORMED'] = pt.fit_transform(train_df[['DELIVERY_DELAY_DAYS']])
test_df['DELAY_TRANSFORMED'] = pt.transform(test_df[['DELIVERY_DELAY_DAYS']])

# VIF-cleaned feature list (ml-engineer review: dropped PRIMARY_PRODUCT_CATEGORY_NAME
# [VIF ~1e25 vs its English counterpart] and AVG_DISTANCE_KM [VIF ~1,120 vs MAX_DISTANCE_KM])
exclude_cols = [
    'ORDER_ID', 'ORDER_PURCHASE_TIMESTAMP',
    'ORDER_DELIVERED_CUSTOMER_DATE', 'ORDER_ESTIMATED_DELIVERY_DATE',
    'PRIMARY_PRODUCT_ID', 'PRIMARY_SELLER_ID',
    'PRIMARY_PRODUCT_CATEGORY_NAME', 'AVG_DISTANCE_KM',
    'DELIVERY_DELAY_DAYS', 'DELAY_TRANSFORMED',
]
feature_cols_v2 = [c for c in train_df.columns if c not in exclude_cols]
print('Features:', feature_cols_v2)

In [ ]:
# NaN -> None (Oracle NUMBER columns can't hold pandas NaN), push to DB
keep_cols = feature_cols_v2 + ['DELAY_TRANSFORMED', 'ORDER_ID']
train_push = train_df[keep_cols].astype(object).where(pd.notnull(train_df[keep_cols]), None)
test_push = test_df[keep_cols].astype(object).where(pd.notnull(test_df[keep_cols]), None)

for t in ('DDF_TRAIN_FINAL', 'DDF_TEST_FINAL'):
    try:
        oml.drop(table=t)
    except Exception:
        pass  # table didn't exist yet -- fine

train_oml = oml.create(train_push, table='DDF_TRAIN_FINAL')
test_oml = oml.create(test_push, table='DDF_TEST_FINAL')

In [ ]:
# Drop any previous version of the named model, then train + persist the final one
try:
    oml.drop(model='DELIVERY_DELAY_GLM_FINAL')
except Exception:
    pass

mod = oml.glm(mining_function='regression', GLMS_SOLVER='GLMS_SOLVER_QR', GLMS_RIDGE_REGRESSION='GLMS_RIDGE_REG_DISABLE')
mod = mod.fit(train_oml[feature_cols_v2], train_oml['DELAY_TRANSFORMED'], model_name='DELIVERY_DELAY_GLM_FINAL')
print(mod)

**Check before moving on:** should show `CONVERGED = YES`, `VALID_COVARIANCE_MATRIX = YES`,
`RANK_DEFICIENCY = 0`, `R^2 ~ 0.418` (training) -- matches the validated clean baseline
exactly, since this is the same code, just with an explicit persistent `model_name` added.

In [ ]:
# Confirm real test-set performance too (supplemental_cols keeps rows aligned --
# oml.DataFrames are proxies over DB tables with no guaranteed row order)
result = mod.predict(test_oml[feature_cols_v2], supplemental_cols=test_oml[['ORDER_ID', 'DELAY_TRANSFORMED']])
result_df = result.pull()
pred_col = [c for c in result_df.columns if c not in ('ORDER_ID', 'DELAY_TRANSFORMED')][0]

print('Test RMSE:', mean_squared_error(result_df['DELAY_TRANSFORMED'], result_df[pred_col]) ** 0.5)
print('Test R^2:', r2_score(result_df['DELAY_TRANSFORMED'], result_df[pred_col]))
print('Expected: RMSE~0.699, R^2~0.619 (matches the validated clean baseline)')

In [ ]:
# Confirm the model actually persisted as a named, permanent object
check = oml.sync(query="SELECT model_name, mining_function, algorithm FROM user_mining_models WHERE model_name = 'DELIVERY_DELAY_GLM_FINAL'")
check.pull()